# Build Mamba-SSM wheel for CUDA 12.2 + PTX

This notebook clones the public repo, installs the build dependencies, and builds a wheel with PTX fallback enabled through `TORCH_CUDA_ARCH_LIST`.

It assumes a Colab GPU runtime with a CUDA 12.x-capable PyTorch install.

In [ ]:
import os
import subprocess

REPO_URL = os.environ.get("REPO_URL", "https://github.com/davidkny22/efficient-mamba-ssm.git")
REPO_DIR = "/content/mamba"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("repo:", os.getcwd())

In [ ]:
!python -m pip install --upgrade pip setuptools wheel ninja packaging
!python - <<'PY'
import torch
print('torch:', torch.__version__)
print('torch cuda:', torch.version.cuda)
PY

In [ ]:
import os

# Force a source build and add PTX fallback for Ada/Hopper-class cards.
os.environ["MAMBA_FORCE_BUILD"] = "TRUE"
os.environ["MAMBA_FORCE_CXX11_ABI"] = "FALSE"
os.environ["MAMBA_LOCAL_VERSION"] = "cu122ptx"
os.environ["MAX_JOBS"] = "2"
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.0;8.6;8.9;9.0+PTX"

!python setup.py --name

!python setup.py bdist_wheel --dist-dir dist
!ls -lh dist

In [ ]:
import glob
print(glob.glob('dist/*.whl'))